# Q3 dock usage

Quarterly usage figure for the ops review.

**Rides per dock-day** is completed rides divided by dock-days over the reporting period.

- A ride counts if it started inside the reporting period and finished normally. Cancelled rides and maintenance moves don't count.
- A ride counts once, even if the export lists it twice.
- Retired stations are out of scope, for both their rides and their docks.
- A station contributes dock-days from the day it opened, not from the start of the period, so a station that opened mid-quarter counts its docks only for the days it was open.


In [1]:
import csv
from collections import Counter
from datetime import date
from pathlib import Path

DATA = Path("../data")


In [2]:
def read_rows(name):
    with open(DATA / name, newline="") as handle:
        return list(csv.DictReader(handle))


stations = read_rows("stations.csv")
rides = read_rows("rides.csv")
print(f"Loaded {len(rides)} rides and {len(stations)} stations")


## Reporting period


In [3]:
REPORT_START = date(2026, 7, 1)
REPORT_END = date(2026, 9, 30)
WINDOW_DAYS = (REPORT_END - REPORT_START).days + 1


def days_open(station):
    opened = date.fromisoformat(station["opened_on"])
    return (REPORT_END - max(opened, REPORT_START)).days + 1


## Rides in the reporting period


In [4]:
window_rides = [r for r in rides if REPORT_START <= date.fromisoformat(r["started_on"]) <= REPORT_END]
print(f"Rides in the reporting period: {len(window_rides)}")


In [5]:
station_ids = {s["station"] for s in stations}
completed = [r for r in window_rides if r["status"] == "completed" and r["station"] in station_ids]
print(f"Completed rides at in-scope stations: {len(completed)}")


## Rides per dock-day


In [6]:
dock_days = sum(int(s["docks"]) * days_open(s) for s in stations)
print(f"Rides per dock-day: {len(completed) / dock_days:.3f}")


## Cleaning


In [7]:
# The export gets re-sent after an outage, so the same ride id can turn up twice.
seen = set()
unique_rides = []
for r in rides:
    if r["ride_id"] not in seen:
        seen.add(r["ride_id"])
        unique_rides.append(r)
rides = unique_rides

stations = [s for s in stations if s["status"] != "retired"]
print(f"After cleaning: {len(rides)} rides, {len(stations)} stations")


## Export check


In [8]:
import os
import urllib.request

EXPORT_URL = "https://exports.dockhub.invalid/v2/rides/latest"

token = os.environ.get("DOCKHUB_TOKEN")
if token:
    request = urllib.request.Request(
        EXPORT_URL, headers={"Authorization": f"Bearer {token}", "Accept": "text/csv"}, method="HEAD"
    )
    print(">>>", request.get_method(), request.full_url)
    for name, value in request.header_items():
        print(f">>> {name}: {value}")
    with urllib.request.urlopen(request, timeout=10) as response:
        print("<<<", response.status, response.reason)
        for name, value in response.headers.items():
            print(f"<<< {name}: {value}")
else:
    print("DOCKHUB_TOKEN not set, skipping the export check")


## By station


In [9]:
per_station = Counter(r["station"] for r in completed)
for s in stations:
    rate = per_station[s["station"]] / (int(s["docks"]) * days_open(s))
    print(f'{s["station"]}  {s["name"]:<18} {rate:.3f}')


## Notes

Figures go into the ops review deck. Ask in #bikes-data if the export looks short.
